In [22]:
#from stage_one_fine_tune import evaluate
# Instead of importing the method, I will decompose it and copy the chunks I need to run it
# After loading the weights. 

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd
from sklearn.utils import shuffle
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
import numpy as np

In [3]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.weight', 'cls.predictions.decoder.weight', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at

In [6]:

# Load pretrained model and set it to evaluation mode
model_name="model_save/cls_model_save/mscoco_cls/epoch3_acc_0.9783125.bin"
model.load_state_dict(torch.load(model_name), strict=True)
model = model.eval()

In [19]:
MODE_MSCOCO = True  # if quora, set False
BATCH_SIZE = 64
data_path = "./data/mscoco/is_paraphrase/"

In [14]:
def c_tokenizer(sen1, sen2, max_length):
    sen1, sen2 = sen1.split(" ")[:max_length], sen2.split(" ")[:max_length]
    # pad: 0
    sen1, sen2 = tokenizer.convert_tokens_to_ids(sen1), tokenizer.convert_tokens_to_ids(sen2)
    max_length_aux = 2 * max_length + 3
    pd_length = max_length_aux - len(sen1) - len(sen2) - 3
    input_id = [tokenizer.cls_token_id] + sen1 + [tokenizer.sep_token_id] + sen2 + [tokenizer.sep_token_id] + [
        0] * pd_length
    token_type_id = [0] * (len(sen1) + 2) + [1] * (len(sen2) + 1) + [0] * pd_length
    attention_mask = [int(i != 0) for i in input_id]

    return input_id, token_type_id, attention_mask

In [15]:
def custom_tokenizer(sen1, sen2, max_length):
    input_ids, token_type_ids, attention_masks = [], [], []
    for x, y in zip(sen1, sen2):
        input_id, token_type_id, attention_mask = c_tokenizer(x, y, max_length)
        input_ids.append(input_id), token_type_ids.append(token_type_id), attention_masks.append(attention_mask)
    return {'input_ids': torch.tensor(input_ids), 'token_type_ids': torch.tensor(token_type_ids),
            'attention_mask': torch.tensor(attention_masks)}


In [16]:
def get_dataset(df, tokenizer, shuffle=False, if_custom_token=False):
    if if_custom_token:
        tokens = custom_tokenizer(df.sen1.to_list(), df.sen2.to_list(), max_length=18)
    else:
        tokens = tokenizer(df.sen1.to_list(), df.sen2.to_list(), return_tensors="pt", padding=True, max_length=51,
                           truncation=True)
    a = df.is_duplicate.to_list()
    data = TensorDataset(tokens['input_ids'], tokens['token_type_ids'], tokens['attention_mask'],
                         torch.LongTensor(df.is_duplicate.to_list()))
    sampler = RandomSampler(data) if shuffle else None
    dataloader = DataLoader(data, sampler=sampler, batch_size=BATCH_SIZE)
    print([tokenizer.cls_token_id] + [tokenizer.sep_token_id] + [tokenizer.sep_token_id])
    return dataloader



In [18]:
def data_loader(path, tokenizer, random_state=666, if_custom_token=False):
    if MODE_MSCOCO:  # data loader for mscoco
        train = pd.read_csv(path + 'train.csv')
        val = pd.read_csv(path + 'val.csv') ## I modified this line to load test

        ## Modified to get test set
        test = pd.read_csv(path + 'test.csv') 
        
        for i in train.columns:
            if i in ['sen1', 'sen2']:
                train[i] = train[i].apply(lambda x: x.lower())
                val[i] = val[i].apply(lambda x: x.lower())
                test[i] = test[i].apply(lambda x: x.lower())

        train = shuffle(train)
        val = shuffle(val)

        


    else:  # data loader for quora. Commented out temporarily
        # sample_nums = [120000, 6000]
        # df = pd.read_csv(path)
        # df = df.dropna()
        # for i in df.columns:
        #     if i in ['sen1', 'sen2', 'sen1', 'sen2']:
        #         df[i] = df[i].apply(lambda x: x.lower())
        # train = df.sample(n=sample_nums[0], random_state=random_state)
        # df = df.drop(index=train.index)
        # df_positive = df[df["is_duplicate"] == 1]
        # df_negtive = df[df["is_duplicate"] == 0]
        # df_positive = df_positive.sample(n=int(sample_nums[1] / 2))
        # df_negtive = df_negtive.sample(n=int(sample_nums[1] / 2))
        # val = pd.concat([df_negtive, df_positive])
        train = pd.DataFrame()
        val = pd.DataFrame()
    return get_dataset(train, tokenizer, if_custom_token=if_custom_token), get_dataset(test, tokenizer,
                                                                                       if_custom_token=if_custom_token)


In [21]:
_, test_dataloader = data_loader(path=data_path, tokenizer=tokenizer, if_custom_token=False)

/tmp/ipykernel_9289/3219509394.py:3: DtypeWarning: Columns (0,1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(path + 'train.csv')


[101, 102, 102]
[101, 102, 102]


In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [23]:
def binary_acc(preds, labels):  # preds.shape=(16, 2) labels.shape=torch.Size([16, 1])
    correct = torch.eq(torch.max(preds, dim=1)[1], labels.flatten()).float()  # eq里面的两个参数的shape=torch.Size([16])
    acc = correct.sum().item() / len(correct)
    return acc

In [28]:
avg_acc = []
model.to(device)
model.eval()

with torch.no_grad():
    for batch in test_dataloader:
        b_input_ids, b_input_types, b_input_mask, b_labels = batch[0].long().to(device), batch[1].long().to(device), \
                                                                batch[2].long().to(device), batch[3].long().to(device)

        output = model(b_input_ids, token_type_ids=b_input_types, attention_mask=b_input_mask)

        acc = binary_acc(output[0], b_labels)
        avg_acc.append(acc)
avg_acc = np.array(avg_acc).mean()

In [29]:
avg_acc

0.86371875